# E2FGVI-HQ CUDA benchmark (FP32)
Giữ nguyên thuật toán/cấu hình CPU thắng. Chạy smoke 3 frame, rồi 48 frame; full video mặc định bị khóa.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Hãy chọn Runtime > Change runtime type > GPU rồi reconnect'
gpu = torch.cuda.get_device_properties(0)
print({
    'gpu': gpu.name,
    'cuda_version': torch.version.cuda,
    'pytorch_version': torch.__version__,
    'vram_gb': round(gpu.total_memory / 1024**3, 2),
})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Source lấy từ GitHub; Drive chỉ giữ dữ liệu lớn và kết quả.
REPO_URL = 'https://github.com/duchieu34/xoa-logo.git'
BRANCH = 'research/e2fgvi-hq-colab-gpu'
DRIVE_ROOT = '/content/drive/MyDrive/veo'
VIDEO_IN_DRIVE = f'{DRIVE_ROOT}/ft-vid-23.mp4'
CHECKPOINT_IN_DRIVE = f'{DRIVE_ROOT}/E2FGVI-HQ-CVPR22.pth'
PROJECT_DIR = '/content/Xoa-logo-video'
GPU_RESULTS_IN_DRIVE = f'{DRIVE_ROOT}/results/e2fgvi_colab_gpu'

In [ ]:
import os
import subprocess
from pathlib import Path
for required in (VIDEO_IN_DRIVE, CHECKPOINT_IN_DRIVE):
    assert Path(required).is_file(), f'Missing: {required}'
project = Path(PROJECT_DIR)
if (project / '.git').is_dir():
    subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only', 'origin', BRANCH], check=True)
elif project.exists():
    raise RuntimeError(f'{PROJECT_DIR} tồn tại nhưng không phải Git repository; hãy xóa runtime rồi chạy lại')
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
print('Project:', Path.cwd())
source = Path('research/e2fgvi_hq/benchmark.py').read_text(encoding='utf-8')
assert '--device' in source, 'Nhánh Git hiện tại chưa có hỗ trợ --device'

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
# Không cài requirements-ai.txt vì file đó cố ý pin bản PyTorch CPU.
!pip install -q 'numpy>=1.26,<3' 'opencv-python-headless>=4.10,<5' psutil==7.0.0 gdown==5.2.0 pytest
!mkdir -p samples research/e2fgvi_hq/checkpoints third_party
!cp "{VIDEO_IN_DRIVE}" samples/ft-vid-23.mp4
!cp "{CHECKPOINT_IN_DRIVE}" research/e2fgvi_hq/checkpoints/E2FGVI-HQ-CVPR22.pth
![ -d third_party/E2FGVI/.git ] || git clone https://github.com/MCG-NKU/E2FGVI.git third_party/E2FGVI
!git -C third_party/E2FGVI checkout 709cbe319edc21b8a365a28e14cba595a93d62cf

In [ ]:
# Smoke test thực tế: 3 frame, FP32 CUDA, cùng crop/mask/temporal config.
!python -m research.e2fgvi_hq.benchmark --device cuda --start-frame 130 --frames 3 --crop-size 192 --neighbor-stride 5 --reference-step 10 --aggregation legacy_average --threads 4 --skip-baselines --output-dir /content/e2fgvi_gpu_smoke --report /content/e2fgvi_gpu_smoke_report.json

In [ ]:
import json
smoke = json.loads(Path('/content/e2fgvi_gpu_smoke_report.json').read_text())
assert smoke['runtime']['device'] == 'cuda'
assert smoke['inference_windows']['outside_mask_max_absolute_change'] == 0
print(smoke['runtime'])
SMOKE_OK = True

In [ ]:
# Benchmark chuẩn: frame 108–155 (48 frame). Chỉ chạy sau smoke thành công.
assert SMOKE_OK
!python -m research.e2fgvi_hq.benchmark --device cuda --start-frame 108 --frames 48 --crop-size 192 --neighbor-stride 5 --reference-step 10 --aggregation legacy_average --threads 4 --skip-baselines --output-dir /content/e2fgvi_gpu_48 --report /content/e2fgvi_gpu_48_report.json

In [ ]:
import shutil
gpu48 = json.loads(Path('/content/e2fgvi_gpu_48_report.json').read_text())
runtime = gpu48['runtime']
print({
    'gpu_s_per_frame': runtime['inference_seconds_per_output_frame'],
    'gpu_fps': runtime['inference_output_fps'],
    'peak_vram_mb': runtime['peak_vram_mb'],
    'speedup_vs_cpu_48': round(2.487 / runtime['inference_seconds_per_output_frame'], 2),
    'transitions': {k: v for k, v in gpu48['metrics']['e2fgvi_hq'].items() if 'transition_' in k},
})
Path(GPU_RESULTS_IN_DRIVE).mkdir(parents=True, exist_ok=True)
shutil.copy2('/content/e2fgvi_gpu_48_report.json', f'{GPU_RESULTS_IN_DRIVE}/benchmark_48_fp32.json')
shutil.make_archive('/content/e2fgvi_gpu_48_diagnostics', 'zip', '/content/e2fgvi_gpu_48')
shutil.copy2('/content/e2fgvi_gpu_48_diagnostics.zip', GPU_RESULTS_IN_DRIVE)
GPU_48_OK = True

In [ ]:
# Chỉ đổi thành True sau khi đã xem diagnostics 48 frame và chấp nhận chất lượng.
RUN_FULL_VIDEO = False
if RUN_FULL_VIDEO:
    assert GPU_48_OK
    !python -m research.e2fgvi_hq.full_video_validation --device cuda --video samples/ft-vid-23.mp4 --checkpoint research/e2fgvi_hq/checkpoints/E2FGVI-HQ-CVPR22.pth --crop-size 192 --neighbor-stride 5 --reference-step 10 --aggregation legacy_average --threads 4 --output-dir /content/e2fgvi_gpu_full --report /content/e2fgvi_gpu_full_report.json
    shutil.copy2('/content/e2fgvi_gpu_full_report.json', f'{GPU_RESULTS_IN_DRIVE}/full_192_fp32.json')
    shutil.copy2('/content/e2fgvi_gpu_full/ft-vid-23_e2fgvi_hq_cuda_full.mp4', GPU_RESULTS_IN_DRIVE)
else:
    print('Full 192-frame run is locked. Review the 48-frame report/diagnostics first.')

## FP16
Chưa được triển khai trong notebook này. Chỉ mở experiment FP16 riêng sau khi FP32 CUDA chạy đúng và output tương đương CPU.